# Step 6.0 — Sanity Check of the Final Model (LR, 6 months)

## Objective
Before proceeding to the explainability chapter (XAI), validate the **operating point** of the final model:
- the model (Logistic Regression) produces probabilities `P(rapid)`
- the binary decision (rapid/slow) depends on the **threshold**

This notebook verifies whether the initially proposed threshold is **operationally reasonable**, and documents the trade-off between:
- **FN (missed rapids)** — most critical error if we want high sensitivity
- **FP (false alarms)** — operational cost (excessive alerts)

## Inputs
- `01_data/processed/dataset_6m_v1.csv`
- `models/final_lr_6m.joblib`
- `models/final_lr_6m_metadata.json`

## Outputs (for methodological evidence)
- `04_outputs/tables/step6_sanity_threshold_checks.csv`
- `04_outputs/tables/step6_sanity_threshold_curve.csv`
- (optional) `04_outputs/figures/step6_sanity_f2_vs_threshold.png`

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Note:</b> this sanity check is not meant to "improve performance" or estimate generalisation.
It validates that the chosen threshold produces a coherent decision regime before generating XAI explanations.
</div>


## 1) Load final model and data (6m)

We load:
- the final 6-month dataset (features @t0 + slope_180d)
- the trained pipeline (preprocessing + Logistic Regression)
- the metadata (includes `slope_cutoff_30pct` and the initial threshold)

Then:
- we reconstruct `y_true` (rapid/slow) from the slope cut-off
- we compute probabilities `P(rapid)` for all patients


In [ ]:
import os
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, fbeta_score

DATASET_6M = os.path.join("..", "01_data", "processed", "dataset_6m_v1.csv")
MODEL_PATH = os.path.join("..", "models", "final_lr_6m.joblib")
META_PATH  = os.path.join("..", "models", "final_lr_6m_metadata.json")

df = pd.read_csv(DATASET_6M)
pipe = joblib.load(MODEL_PATH)

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

slope_col = meta["slope_col"]
slope_cutoff = meta["slope_cutoff_30pct"]
thr_final = meta["decision_threshold"]
feat_cols = meta["feature_cols"]

# target "rapid" (same rule as final training)
y_true = (df[slope_col] <= slope_cutoff).astype(int).to_numpy()

X = df[feat_cols].copy()
proba = pipe.predict_proba(X)[:, 1]

print("N:", len(df), "| rapid% true:", y_true.mean(), "| slope_cutoff:", slope_cutoff, "| thr_final:", thr_final)
print("proba min/mean/max:", proba.min(), proba.mean(), proba.max())


## 2) Evaluate different thresholds (operating point)

A probabilistic model needs a threshold to convert probability into a decision.
Here we compare 3 thresholds:

1) **metadata threshold** (initial)  
2) **0.50** (typical "neutral" point)  
3) **threshold that produces ~30% predicted positives**  
   (useful as a reference, since the prevalence of rapid is ~30%)

### Metrics used
- **pred_pos_rate**: percentage predicted as rapid (operational load)
- **precision**: among those predicted as rapid, how many are rapid (alert quality)
- **recall**: among true rapids, how many were detected (sensitivity)
- **F2**: gives more weight to recall (aligned with "do not miss rapids")
- **TN/FP/FN/TP**: confusion matrix (direct operational impact)

<b>Clinical interpretation:</b> a low threshold tends to reduce FN but increases FP; a high threshold does the opposite.


In [ ]:
def eval_at_threshold(y_true, proba, thr, beta=2):
    y_pred = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()

    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    f2   = fbeta_score(y_true, y_pred, beta=beta, zero_division=0)

    return {
        "thr": float(thr),
        "pred_pos_rate": float(y_pred.mean()),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "precision": float(prec),
        "recall": float(rec),
        "F1": float(f1),
        "F2": float(f2),
    }

# threshold that produces ~30% predicted positives (for comparison)
thr_30pct_pred = float(np.quantile(proba, 1 - 0.30))

checks = [
    eval_at_threshold(y_true, proba, thr_final),
    eval_at_threshold(y_true, proba, 0.50),
    eval_at_threshold(y_true, proba, thr_30pct_pred),
]

pd.DataFrame(checks)


## 3) Threshold curve → (precision, recall, F2, positive rate)

Instead of choosing a threshold "blindly", we sweep a range of thresholds (e.g., 0.05–0.95).
This allows us to:

- observe how **pred_pos_rate** decreases as the threshold increases
- quantify the loss of **recall** when reducing alerts
- find a final threshold that represents an operational compromise

<b>Expected result:</b> select a final threshold that preserves high recall
without turning the model into an "alarm that flags almost everyone as rapid".


In [ ]:
ths = np.linspace(0.05, 0.95, 19)
rows = [eval_at_threshold(y_true, proba, t) for t in ths]
curve = pd.DataFrame(rows)[["thr","pred_pos_rate","precision","recall","F2"]]
curve

## 4) Save sanity check results (decision evidence)

We save:
- comparison table (checks) with the 3 main thresholds
- full curve table (curve)
- (optional) figure F2 vs threshold

This serves as reproducible evidence that the final threshold was chosen
based on an explicitly documented trade-off.


In [ ]:
import os

OUT_TABLES = os.path.join("..", "04_outputs", "tables")
os.makedirs(OUT_TABLES, exist_ok=True)

checks_path = os.path.join(OUT_TABLES, "step6_sanity_threshold_checks.csv")
curve_path  = os.path.join(OUT_TABLES, "step6_sanity_threshold_curve.csv")

pd.DataFrame(checks).to_csv(checks_path, index=False)
curve.to_csv(curve_path, index=False)

print("Saved:", checks_path)
print("Saved:", curve_path)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(curve["thr"], curve["F2"], marker="o")
plt.xlabel("threshold")
plt.ylabel("F2")
plt.title("Sanity check — F2 vs threshold (6m, Logistic Regression)")
plt.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = os.path.join("..", "04_outputs", "figures", "step6_sanity_f2_vs_threshold.png")
os.makedirs(os.path.dirname(fig_path), exist_ok=True)
plt.savefig(fig_path, dpi=300)
plt.show()
print("Saved:", fig_path)


## 5) Update the final threshold (without re-training the model)

The threshold does not change the Logistic Regression weights — it only defines the rule:
> `rapid = 1` if `P(rapid) ≥ threshold`

Therefore:
- **no re-training is needed**
- we only update `final_lr_6m_metadata.json`
- the `.joblib` pipeline remains the same

<b>Justification:</b> separating the "model" (probability) from the "operating point" (threshold)
allows adjusting the decision to the clinical cost (FN vs FP) without altering the estimator.


In [ ]:
import json
import os

META_PATH = os.path.join("..", "models", "final_lr_6m_metadata.json")

with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

meta["decision_threshold_old"] = meta.get("decision_threshold", None)
meta["decision_threshold"] = 0.40
meta["decision_threshold_reason"] = "sanity-check trade-off: keep recall high while reducing predicted positives"

with open(META_PATH, "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print("Updated:", META_PATH)
print("old -> new:", meta["decision_threshold_old"], "->", meta["decision_threshold"])


## ✅ Takeaways (linking to Step 6 — XAI)

- The sanity check confirmed that the initial threshold was too aggressive
  (predicted positive rate too high).
- A more operational final threshold was selected,
  preserving high recall while reducing excessive alerts.
- With the final threshold fixed, we proceed to XAI:
  - global explanation (which features influence risk)
  - local explanation (TP/FP/FN/TN) with consistent decisions.
